In [2]:
from pathlib import Path
import numpy as np
from PIL import Image


class StressColorConverter:
    def __init__(
        self,
        input_dir=r"E:\RFNS\CODE-R2",
        output_dir=r"E:\RFNS\CODE-R2\converted_blue",
        background_threshold=245,
        alpha_threshold=5,
        saturation_min=0.12,
        value_min=0.08,
        low_color=(198, 226, 245),
        high_color=(8, 48, 107),
        gamma=1.0,
        transparent_background_to_white=True,
        make_colorbar=True,
        colorbar_width=70,
        colorbar_height=420,
        debug=True,
    ):
        # Adjustable parameters
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.background_threshold = background_threshold
        self.alpha_threshold = alpha_threshold
        self.saturation_min = saturation_min
        self.value_min = value_min
        self.low_color = np.array(low_color, dtype=np.float32)
        self.high_color = np.array(high_color, dtype=np.float32)
        self.gamma = gamma
        self.transparent_background_to_white = transparent_background_to_white
        self.make_colorbar = make_colorbar
        self.colorbar_width = colorbar_width
        self.colorbar_height = colorbar_height
        self.debug = debug

        self.output_dir.mkdir(parents=True, exist_ok=True)

    def run(self):
        files = sorted(self.input_dir.glob("*.png"))

        if not files:
            print(f"No PNG files found in: {self.input_dir}")
            return

        print(f"Found {len(files)} PNG files.")

        for file_path in files:
            self.convert_one(file_path)

        if self.make_colorbar:
            self.save_colorbar()

        print(f"Saved converted images to: {self.output_dir}")

    def convert_one(self, file_path):
        image = Image.open(file_path)
        original_mode = image.mode
        has_alpha = self._has_alpha(image)

        rgba = np.asarray(image.convert("RGBA")).astype(np.float32)

        rgb = rgba[..., :3]
        alpha = rgba[..., 3]

        hsv = self._rgb_to_hsv(rgb)

        h = hsv[..., 0]
        s = hsv[..., 1]
        v = hsv[..., 2]

        transparent_mask = alpha <= self.alpha_threshold
        white_mask = np.all(rgb >= self.background_threshold, axis=-1)
        background_mask = transparent_mask | white_mask

        # Convert only saturated contour pixels. Preserve gray/black boundary lines.
        color_mask = (
            (~background_mask)
            & (s >= self.saturation_min)
            & (v >= self.value_min)
        )

        level = self._abaqus_hue_to_level(h)
        level = np.clip(level, 0.0, 1.0)
        level = level ** self.gamma

        converted_rgb = rgb.copy()
        blue_rgb = self._level_to_blue(level)

        converted_rgb[color_mask] = blue_rgb[color_mask]

        if self.transparent_background_to_white:
            output_rgba = self._make_white_background(converted_rgb, alpha, background_mask)
        else:
            output_rgba = self._preserve_alpha_background(converted_rgb, alpha, white_mask, transparent_mask)

        output = np.clip(output_rgba, 0, 255).astype(np.uint8)
        output_image = Image.fromarray(output, mode="RGBA")

        output_path = self.output_dir / f"{file_path.stem}_blue.png"
        output_image.save(output_path)

        if self.debug:
            self._print_image_info(
                file_path=file_path,
                original_mode=original_mode,
                size=image.size,
                has_alpha=has_alpha,
                transparent_mask=transparent_mask,
                white_mask=white_mask,
                background_mask=background_mask,
                color_mask=color_mask,
                s=s,
                h=h,
                output_path=output_path,
            )

    def save_colorbar(self):
        border = 2
        w = self.colorbar_width
        h = self.colorbar_height

        canvas = np.ones((h + 2 * border, w + 2 * border, 4), dtype=np.float32) * 255.0
        canvas[..., 3] = 255.0

        levels = np.linspace(1.0, 0.0, h, dtype=np.float32)[:, None]
        levels = np.repeat(levels, w, axis=1)

        colors = self._level_to_blue(levels)
        alpha = np.ones((h, w, 1), dtype=np.float32) * 255.0
        colorbar = np.concatenate([colors, alpha], axis=-1)

        canvas[border:border + h, border:border + w, :] = colorbar

        # Border
        canvas[:border, :, :3] = 0
        canvas[-border:, :, :3] = 0
        canvas[:, :border, :3] = 0
        canvas[:, -border:, :3] = 0

        output = np.clip(canvas, 0, 255).astype(np.uint8)
        output_image = Image.fromarray(output, mode="RGBA")

        output_path = self.output_dir / "blue_colorbar.png"
        output_image.save(output_path)

        if self.debug:
            print(f"Colorbar saved: {output_path.name}")

    def _make_white_background(self, converted_rgb, alpha, background_mask):
        alpha_norm = alpha[..., None] / 255.0

        white = np.ones_like(converted_rgb) * 255.0
        composite_rgb = converted_rgb * alpha_norm + white * (1.0 - alpha_norm)

        composite_rgb[background_mask] = 255.0

        output_alpha = np.ones(alpha.shape, dtype=np.float32) * 255.0
        output_rgba = np.concatenate([composite_rgb, output_alpha[..., None]], axis=-1)

        return output_rgba

    def _preserve_alpha_background(self, converted_rgb, alpha, white_mask, transparent_mask):
        converted_rgb[white_mask] = 255.0
        converted_rgb[transparent_mask] = 255.0

        output_rgba = np.concatenate([converted_rgb, alpha[..., None]], axis=-1)

        return output_rgba

    def _print_image_info(
        self,
        file_path,
        original_mode,
        size,
        has_alpha,
        transparent_mask,
        white_mask,
        background_mask,
        color_mask,
        s,
        h,
        output_path,
    ):
        total_pixels = color_mask.size
        transparent_pixels = int(np.sum(transparent_mask))
        white_pixels = int(np.sum(white_mask & (~transparent_mask)))
        background_pixels = int(np.sum(background_mask))
        converted_pixels = int(np.sum(color_mask))
        kept_non_bg_pixels = int(np.sum((~background_mask) & (~color_mask)))

        transparent_ratio = transparent_pixels / total_pixels * 100.0
        white_ratio = white_pixels / total_pixels * 100.0
        background_ratio = background_pixels / total_pixels * 100.0
        converted_ratio = converted_pixels / total_pixels * 100.0

        if converted_pixels > 0:
            hue_used = h[color_mask]
            sat_used = s[color_mask]
            hue_min = float(np.percentile(hue_used, 1))
            hue_max = float(np.percentile(hue_used, 99))
            sat_mean = float(np.mean(sat_used))
            hue_text = f"hue_p1_p99=({hue_min:.1f}, {hue_max:.1f}), sat_mean={sat_mean:.3f}"
        else:
            hue_text = "hue_p1_p99=N/A, sat_mean=N/A"

        print(
            f"{file_path.name} | "
            f"size={size}, mode={original_mode}, alpha={has_alpha}, "
            f"transparent={transparent_pixels}({transparent_ratio:.2f}%), "
            f"white_bg={white_pixels}({white_ratio:.2f}%), "
            f"all_bg={background_pixels}({background_ratio:.2f}%), "
            f"converted={converted_pixels}({converted_ratio:.2f}%), "
            f"kept_non_bg={kept_non_bg_pixels}, "
            f"{hue_text} -> {output_path.name}"
        )

    def _has_alpha(self, image):
        if image.mode in ("RGBA", "LA"):
            return True
        if image.mode == "P" and "transparency" in image.info:
            return True
        return False

    def _abaqus_hue_to_level(self, hue):
        """
        Abaqus rainbow order:
        blue/cyan/green/yellow/red = low to high.
        Return normalized level: 0 low, 1 high.
        """
        level = np.zeros_like(hue, dtype=np.float32)

        main_mask = hue <= 240.0
        level[main_mask] = (240.0 - hue[main_mask]) / 240.0

        red_wrap_mask = hue >= 300.0
        level[red_wrap_mask] = 1.0

        purple_mask = (hue > 240.0) & (hue < 300.0)
        level[purple_mask] = 0.0

        return level

    def _level_to_blue(self, level):
        """
        level = 0: light blue
        level = 1: dark blue
        """
        level_3d = level[..., None]
        return self.low_color * (1.0 - level_3d) + self.high_color * level_3d

    def _rgb_to_hsv(self, rgb):
        rgb_norm = rgb / 255.0

        r = rgb_norm[..., 0]
        g = rgb_norm[..., 1]
        b = rgb_norm[..., 2]

        maxc = np.max(rgb_norm, axis=-1)
        minc = np.min(rgb_norm, axis=-1)
        delta = maxc - minc

        h = np.zeros_like(maxc, dtype=np.float32)

        r_mask = (maxc == r) & (delta != 0)
        g_mask = (maxc == g) & (delta != 0)
        b_mask = (maxc == b) & (delta != 0)

        h[r_mask] = 60.0 * (((g[r_mask] - b[r_mask]) / delta[r_mask]) % 6.0)
        h[g_mask] = 60.0 * (((b[g_mask] - r[g_mask]) / delta[g_mask]) + 2.0)
        h[b_mask] = 60.0 * (((r[b_mask] - g[b_mask]) / delta[b_mask]) + 4.0)

        s = np.zeros_like(maxc, dtype=np.float32)
        nonzero_mask = maxc != 0
        s[nonzero_mask] = delta[nonzero_mask] / maxc[nonzero_mask]

        v = maxc

        hsv = np.stack([h, s, v], axis=-1)
        return hsv


converter = StressColorConverter(
    input_dir=r"E:\RFNS\CODE-R2",
    output_dir=r"E:\RFNS\CODE-R2\converted_blue",
    background_threshold=245,
    alpha_threshold=5,
    saturation_min=0.12,
    value_min=0.08,
    low_color=(198, 226, 245),
    high_color=(8, 48, 107),
    gamma=1.0,
    transparent_background_to_white=True,
    make_colorbar=True,
    colorbar_width=70,
    colorbar_height=420,
    debug=True,
)

converter.run()

Found 5 PNG files.
1.png | size=(1193, 2475), mode=RGBA, alpha=True, transparent=0(0.00%), white_bg=2603434(88.17%), all_bg=2603434(88.17%), converted=298068(10.09%), kept_non_bg=51173, hue_p1_p99=(73.6, 244.1), sat_mean=0.855 -> 1_blue.png
2.png | size=(841, 1562), mode=RGB, alpha=False, transparent=1220315(92.90%), white_bg=0(0.00%), all_bg=1220315(92.90%), converted=93327(7.10%), kept_non_bg=0, hue_p1_p99=(109.2, 240.0), sat_mean=1.000 -> 2_blue.png
3.png | size=(841, 1562), mode=RGB, alpha=False, transparent=1084865(82.58%), white_bg=0(0.00%), all_bg=1084865(82.58%), converted=228777(17.42%), kept_non_bg=0, hue_p1_p99=(130.6, 240.0), sat_mean=1.000 -> 3_blue.png
4.png | size=(381, 925), mode=RGBA, alpha=True, transparent=307557(87.27%), white_bg=0(0.00%), all_bg=307557(87.27%), converted=44868(12.73%), kept_non_bg=0, hue_p1_p99=(174.1, 240.0), sat_mean=1.000 -> 4_blue.png
5.png | size=(322, 713), mode=RGBA, alpha=True, transparent=209428(91.22%), white_bg=0(0.00%), all_bg=209428(91